In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

Data Reading

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@datalaketanish.dfs.core.windows.net/products")

In [0]:
df.createOrReplaceTempView("products") 

In [0]:
df = df.drop("_rescued_data")
df.limit(5).display()

product_id,product_name,category,brand,price
P0001,Clearly Its,Beauty,Nike,1868.54
P0002,Production Clear,Beauty,Apple,587.13
P0003,Culture Coach,Home,Revlon,1599.24
P0004,Movement Part,Sports,LG,651.71
P0005,Fact Name,Clothing,Samsung,1861.78


Functions

In [0]:
%sql
CREATE Or REPLACE FUNCTION databricks_cata.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE 
LANGUAGE SQL
RETURN p_price * 0.90

In [0]:
%sql
select product_id,price,databricks_cata.bronze.discount_func(price) as discounted_price from products

product_id,price,discounted_price
P0001,1868.54,1681.686
P0002,587.13,528.417
P0003,1599.24,1439.316
P0004,651.71,586.5390000000001
P0005,1861.78,1675.602
P0006,936.36,842.724
P0007,1954.02,1758.618
P0008,1251.26,1126.134
P0009,1247.15,1122.4350000000002
P0010,975.53,877.977


In [0]:

df = df.withColumn("discounted_price",expr("databricks_cata.bronze.discount_func(price)"))
df.limit(5).display()

product_id,product_name,category,brand,price,discounted_price
P0001,Clearly Its,Beauty,Nike,1868.54,1681.686
P0002,Production Clear,Beauty,Apple,587.13,528.417
P0003,Culture Coach,Home,Revlon,1599.24,1439.316
P0004,Movement Part,Sports,LG,651.71,586.5390000000001
P0005,Fact Name,Clothing,Samsung,1861.78,1675.602


In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cata.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS 
$$
    return p_brand.upper()
$$


In [0]:
%sql
SELECT product_id, brand, databricks_cata.bronze.upper_func(brand) AS brand_upper
FROM products

product_id,brand,brand_upper
P0001,Nike,NIKE
P0002,Apple,APPLE
P0003,Revlon,REVLON
P0004,LG,LG
P0005,Samsung,SAMSUNG
P0006,Adidas,ADIDAS
P0007,Adidas,ADIDAS
P0008,Puma,PUMA
P0009,Samsung,SAMSUNG
P0010,Sony,SONY


In [0]:
df.write.mode("overwrite").format("delta").save("abfss://silver@datalaketanish.dfs.core.windows.net/products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  databricks_cata.silver.products_silver 
USING DELTA
LOCATION 'abfss://silver@datalaketanish.dfs.core.windows.net/products'